# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('D:\\DV Module\\dataviz-exercises-Bhushan\\week05\\airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [4]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [5]:
# Task 1 — Overlapping histograms: entire homes vs private rooms

# keep only the two room types worth comparing
two_types = df_cap[df_cap['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

colour_map = {
    'Entire home/apt': '#2E75B6',
    'Private room':    '#70AD47'
}

fig = px.histogram(
    two_types,
    x='price',
    color='room_type',
    barmode='overlay',
    nbins=50,
    opacity=0.6,
    color_discrete_map=colour_map,
    labels={'price': 'Nightly Price (£)', 'count': 'Number of Listings', 'room_type': ''},
    title='Entire homes cluster around £160 — private rooms peak at £80, rarely exceeding £150'
)

# median line per room type — colour matches the histogram bars
for room_type, colour in colour_map.items():
    med = two_types[two_types['room_type'] == room_type]['price'].median()
    fig.add_vline(
        x=med, line_dash='dash', line_color=colour, line_width=2,
        annotation=dict(
            text=f'<b>Median {room_type.split("/")[0].strip()}: £{med:.0f}</b>',
            font=dict(color=colour, size=11, family='Arial'),
            xanchor='left', yanchor='top', xshift=8
        )
    )

# disclose the 95th percentile cap — always annotate data decisions
fig.add_annotation(
    x=0.99, y=0.97, xref='paper', yref='paper',
    text=f'Prices capped at 95th percentile (£{p95:.0f}) to show shape',
    showarrow=False,
    font=dict(size=10, color='#888888', family='Arial'),
    xanchor='right'
)

fig.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='Number of Listings'),
    xaxis=dict(showgrid=False),
    legend=dict(x=0.72, y=0.95, bgcolor='rgba(0,0,0,0)'),
    margin=dict(l=60, r=40, t=65, b=40),
    height=500, width=950
)

fig.show()


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out before plotting

In [6]:
# Task 2 — Horizontal box plot: listing activity by borough

# filter out inactive/new listings with zero reviews
active = df[df['reviews_per_month'] > 0].copy()

# sort boroughs by median activity — most active will sit at top
med_order = (
    active.groupby('neighbourhood')['reviews_per_month']
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

# identify the two most active boroughs for highlighting
top2 = med_order[-2:]

colour_map = {b: ('#E63946' if b in top2 else '#DDDDDD') for b in med_order}

fig = px.box(
    active,
    x='reviews_per_month',
    y='neighbourhood',
    color='neighbourhood',
    color_discrete_map=colour_map,
    orientation='h',
    points='outliers',
    category_orders={'neighbourhood': med_order},
    labels={'reviews_per_month': 'Reviews per Month (proxy for bookings)', 'neighbourhood': ''},
    title=f'{top2[1]} and {top2[0]} lead London for listing activity — turnover is nearly twice the city median'
)

# overall median line gives a single reference point across all boroughs
overall_med = active['reviews_per_month'].median()
fig.add_vline(
    x=overall_med, line_dash='dash', line_color='#555555', line_width=1.5,
    annotation=dict(
        text=f'<b>City median: {overall_med:.1f}</b>',
        font=dict(color='#555555', size=11, family='Arial'),
        xanchor='left', yanchor='bottom', xshift=8
    )
)

fig.update_traces(showlegend=False)

fig.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    xaxis=dict(gridcolor='#EEEEEE', title='Reviews per Month'),
    yaxis=dict(showgrid=False),
    margin=dict(l=180, r=40, t=55, b=40),
    height=700, width=950
)

fig.show()
